# FineBadminton frame cache

Extract frames **once**, on CPU, so GPU quota is never spent decoding video.
Only clips that FBBench references are extracted, not all 3,215.

Same logic as `scripts/extract_frames.py`, adapted for a notebook.

Run order: `DIAGNOSTIC_ONLY = True` first (FBBench only, ~3 min) to learn the
real field names; then `False` with `LIMIT = 20`; then `LIMIT = None`.

In [ ]:
# 1. Config
DIAGNOSTIC_ONLY = True   # True: download FBBench only, dump its schema, skip extraction
FPS = 2          # matches the FineBadminton uniform baseline (~25 frames / rally)
SIZE = 448       # short-side pixels
LIMIT = 20       # None for the full run
WORKERS = 4
JPEG_QUALITY = 88

import pathlib
import shutil
import subprocess
TEMP = pathlib.Path("/kaggle/temp")
FBBENCH_DIR = TEMP / "fbbench"
CLIPS_DIR = TEMP / "clips"
WORK = pathlib.Path("/kaggle/working")
OUT = WORK / "frames"

In [ ]:
# 2. Download FBBench; in a full run also install decord and download the clips
from huggingface_hub import snapshot_download

TEMP.mkdir(parents=True, exist_ok=True)
snapshot_download("Moujuruo/FineBadmintonBenchmark", repo_type="dataset", local_dir=str(FBBENCH_DIR))

if DIAGNOSTIC_ONLY:
    print("DIAGNOSTIC_ONLY: skipping decord install and the clips download")
else:
    subprocess.run(["pip", "install", "-q", "decord"], check=True)
    snapshot_download("Moujuruo/Finebadminton-20K", repo_type="dataset", local_dir=str(CLIPS_DIR))

    # If the clips ship as archives, unpack them so the stem index below can see the videos
    for arc in [p for p in CLIPS_DIR.rglob("*") if p.name.endswith((".zip", ".tar", ".tar.gz", ".tgz"))]:
        print("unpacking", arc.name)
        shutil.unpack_archive(str(arc), str(arc.parent))
        arc.unlink()

subprocess.run(["du", "-sh", *[str(d) for d in (FBBENCH_DIR, CLIPS_DIR) if d.exists()]])

In [ ]:
# 3. Find the clips FBBench references and match them to videos on disk
import json

VIDEO_EXT = {".mp4", ".avi", ".mkv", ".mov", ".webm"}
# "id" last: on a QA record it is usually the question id, not the clip
CANDIDATES = ("clip_id", "video_id", "video", "clip", "vid", "video_name", "id")


def load_records(p: pathlib.Path) -> list:
    text = p.read_text(encoding="utf-8")
    if p.suffix == ".jsonl":
        return [json.loads(l) for l in text.splitlines() if l.strip()]
    return [json.loads(text)]


def iter_dicts(obj):
    """Walk nested JSON: FBBench may nest questions under keys or lists."""
    if isinstance(obj, dict):
        yield obj
        for v in obj.values():
            yield from iter_dicts(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_dicts(v)


def is_qa(r: dict) -> bool:
    """A QA record holds question text; a container like {"questions": [...]} does not."""
    return any("question" in str(k).lower() and isinstance(v, str) for k, v in r.items())


from collections import Counter

wanted: set[str] = set()
sample = None                 # first QA record, else first non-trivial dict
used_field = Counter()        # which candidate supplied the clip id, per record
all_keys = Counter()          # every key seen on QA records
n_qa = 0

files = sorted(list(FBBENCH_DIR.rglob("*.json")) + list(FBBENCH_DIR.rglob("*.jsonl")))
for p in files:
    try:
        records = load_records(p)
    except (OSError, ValueError) as e:
        print(f"skip {p.name}: {e}")
        continue
    for r in iter_dicts(records):
        qa = is_qa(r)
        if qa:
            n_qa += 1
            all_keys.update(r.keys())
        if (sample is None and len(r) > 1) or (qa and (sample is None or not is_qa(sample))):
            sample = r
        for k in CANDIDATES:
            v = r.get(k)
            if isinstance(v, (str, int)):
                wanted.add(pathlib.Path(str(v)).stem)
                used_field[k] += 1
                break

if DIAGNOSTIC_ONLY:
    print("FBBench files:")
    for p in files:
        print(f"  {p.relative_to(FBBENCH_DIR)}  ({p.stat().st_size / 1e6:.1f} MB)")
    print("\nOther files:", sorted({p.suffix for p in FBBENCH_DIR.rglob("*") if p.is_file()} - {".json", ".jsonl"}))
    print(f"\nQA records (dicts with question text): {n_qa}   (paper says 2,563)")
    print("\nCandidate fields that supplied a clip id:", dict(used_field) or "NONE")
    print(f"Distinct clip ids from those fields: {len(wanted)}   e.g. {sorted(wanted)[:5]}")
    print("\nAll keys on QA records (key: count):")
    for k, c in all_keys.most_common():
        print(f"  {k}: {c}")
    print("\nOne full sample record:")
    print(json.dumps(sample, indent=2, ensure_ascii=False))
else:
    have = {p.stem: p for p in CLIPS_DIR.rglob("*") if p.suffix.lower() in VIDEO_EXT}
    matched = sorted(wanted & set(have))
    missing = sorted(wanted - set(have))

    print(f"referenced by FBBench: {len(wanted)}")
    print(f"videos on disk:        {len(have)}")
    print(f"matched:               {len(matched)}")
    if missing:
        print(f"missing, e.g.:         {missing[:5]}")
    if not matched:
        print("\nZERO MATCHED. One full sample record:")
        print(json.dumps(sample, indent=2, ensure_ascii=False))
        print("\nSample video stems:", list(have)[:5])

In [ ]:
# 4. Extract frames with decord, WORKERS processes
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed


def extract_one(video: pathlib.Path, out_dir: pathlib.Path,
                fps: float, size: int, quality: int) -> tuple[str, int, str]:
    """Return (clip_id, n_frames, status)."""
    clip_id = video.stem
    dest = out_dir / clip_id

    if dest.exists() and any(dest.glob("*.jpg")):
        return clip_id, len(list(dest.glob("*.jpg"))), "skipped"

    try:
        import decord
        from PIL import Image

        vr = decord.VideoReader(str(video), num_threads=1)
        src_fps = float(vr.get_avg_fps()) or 25.0
        step = max(1, round(src_fps / fps))
        idx = list(range(0, len(vr), step))
        if not idx:
            return clip_id, 0, "empty"

        batch = vr.get_batch(idx).asnumpy()
        dest.mkdir(parents=True, exist_ok=True)

        for n, (frame, src_i) in enumerate(zip(batch, idx)):
            im = Image.fromarray(frame)
            w, h = im.size
            scale = size / min(w, h)                      # short side -> `size`
            im = im.resize((round(w * scale), round(h * scale)), Image.BICUBIC)
            # timestamp in the filename keeps the cache self-describing
            t_ms = round(src_i / src_fps * 1000)
            im.save(dest / f"{n:03d}_{t_ms:07d}.jpg", quality=quality)

        return clip_id, len(idx), "ok"

    except Exception as e:                                 # noqa: BLE001
        return clip_id, 0, f"error: {type(e).__name__}: {e}"


if DIAGNOSTIC_ONLY:
    print("DIAGNOSTIC_ONLY: skipping extraction")
else:
    todo = [have[c] for c in matched]
    if LIMIT:
        todo = todo[:LIMIT]
    print(f"extracting {len(todo)} clips at {FPS} fps, short side {SIZE}px")

    OUT.mkdir(parents=True, exist_ok=True)
    results, errors = [], []

    # fork, explicitly: functions defined in a notebook cell cannot be pickled
    # by reference under spawn/forkserver
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=mp.get_context("fork")) as pool:
        futs = {pool.submit(extract_one, v, OUT, FPS, SIZE, JPEG_QUALITY): v for v in todo}
        for i, fut in enumerate(as_completed(futs), 1):
            cid, n, status = fut.result()
            results.append({"clip_id": cid, "frames": n, "status": status})
            if status.startswith("error"):
                errors.append((cid, status))
            if i % 50 == 0 or i == len(todo):
                print(f"  {i}/{len(todo)}")

    (OUT / "manifest.json").write_text(json.dumps({
        "fps": FPS, "size": SIZE,
        "clips": len(results),
        "frames": sum(r["frames"] for r in results),
        "missing_videos": missing,
        "results": sorted(results, key=lambda r: r["clip_id"]),
    }, indent=1))

    total = sum(r["frames"] for r in results)
    mb = sum(f.stat().st_size for f in OUT.rglob("*.jpg")) / 1e6
    print(f"\ndone: {len(results)} clips, {total} frames, {mb:.0f} MB")
    if errors:
        print(f"{len(errors)} failed, first few: {errors[:3]}")

In [ ]:
# 5. Keep fbbench with the output, drop the raw clips, report disk usage
if DIAGNOSTIC_ONLY:
    print("DIAGNOSTIC_ONLY: skipping copy and cleanup")
else:
    shutil.copytree(FBBENCH_DIR, WORK / "fbbench", dirs_exist_ok=True)
    shutil.rmtree(CLIPS_DIR, ignore_errors=True)

    subprocess.run(["du", "-sh", str(OUT), str(WORK / "fbbench"), str(WORK)])
    subprocess.run(["df", "-h", str(WORK), str(TEMP)])